In [1]:
!pip install transformers evaluate datasets -q

# STEP 2: Import libraries
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import evaluate
import pandas as pd
import torch

# STEP 3: Load model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
model.eval()

# STEP 4: Load your custom dataset (image paths and ground-truth captions)
data = pd.DataFrame({
    "image_path": [
        r"C:\Users\akra0003\Desktop\JuypterInterior\image1.jpg",
        r"C:\Users\akra0003\Desktop\JuypterInterior\image2.jpg",
    ],
    "reference": [
        ["A bridge over river with small boats", "Bridge"],
        ["a woman sitting on the beach with dog", "woman with dog"]
    ]
})

# STEP 5: Generate predictions
predictions = []

for image_path in data["image_path"]:
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)
    predictions.append(caption)

data["prediction"] = predictions

# STEP 6: Evaluate using BLEU and METEOR
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")

bleu_result = bleu.compute(predictions=data["prediction"].tolist(), references=data["reference"].tolist())
meteor_result = meteor.compute(predictions=data["prediction"].tolist(), references=data["reference"].tolist())

print("\nBLEU Score:", bleu_result)
print("\nMETEOR Score:", meteor_result)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\akra0003\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\akra0003\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\akra0003\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



BLEU Score: {'bleu': 0.45534481052280557, 'precisions': [0.7222222222222222, 0.5, 0.35714285714285715, 0.3333333333333333], 'brevity_penalty': 1.0, 'length_ratio': 4.5, 'translation_length': 18, 'reference_length': 4}

METEOR Score: {'meteor': np.float64(0.8449074074074074)}


In [2]:
# Print predictions and references
for i, row in data.iterrows():
    print(f"\nImage {i+1}: {row['image_path']}")
    print(f"🔹 Prediction: {row['prediction']}")
    print(f"🔸 References: {row['reference']}")


Image 1: C:\Users\akra0003\Desktop\JuypterInterior\image1.jpg
🔹 Prediction: a bridge over a river with boats on it
🔸 References: ['A bridge over river with small boats', 'Bridge']

Image 2: C:\Users\akra0003\Desktop\JuypterInterior\image2.jpg
🔹 Prediction: a woman sitting on the beach with her dog
🔸 References: ['a woman sitting on the beach with dog', 'woman with dog']
